# View offline **ranking** eval (read-only)

Loads CSVs from `recs_job_eval_ranking.py` (frozen pools → rerank + full-catalog `popularity_train` baseline). Does **not** re-score two-tower retrieval.

**Prerequisite:**

1. Retrieval pools: `python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json --examples-parquet artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet`
2. Ranking eval: `python scripts/recs_job_eval_ranking.py configs/recs_job_eval_ranking.json`

Contract: [`docs/recommendation_evaluation_overview.md`](../../docs/recommendation_evaluation_overview.md)

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# "latest_ranking" (default) or a named folder under runs/
EVAL_RUN = "latest_ranking"

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
RUNS_ROOT = REPO_ROOT / "artifacts/recs/offline_eval/runs"
EVAL_DIR = RUNS_ROOT / "latest_ranking" if EVAL_RUN == "latest_ranking" else RUNS_ROOT / EVAL_RUN

if not EVAL_DIR.is_dir():
    raise FileNotFoundError(
        f"Ranking eval dir not found: {EVAL_DIR}\n"
        "Run: python scripts/recs_job_eval_ranking.py configs/recs_job_eval_ranking.json"
    )

PATHS = {
    "ranking_overall": EVAL_DIR / "eval_ranking_overall.csv",
    "ranking_by_slice": EVAL_DIR / "eval_ranking_by_slice.csv",
    "ranking_by_support": EVAL_DIR / "eval_ranking_by_support_bucket.csv",
    "ranking_by_pop_decile": EVAL_DIR / "eval_ranking_by_pop_decile.csv",
    "ranking_pop_delta": EVAL_DIR / "eval_ranking_pop_delta_vs_popularity.csv",
    "ranking_personalization": EVAL_DIR / "eval_ranking_personalization.csv",
    "run_meta": EVAL_DIR / "eval_ranking_run_meta.json",
}
print(f"EVAL_DIR={EVAL_DIR}")

EVAL_DIR=/home/ryanr/workspace/steam_recommendations/artifacts/recs/offline_eval/runs/latest_ranking


In [2]:
def _load_csv(key: str) -> pd.DataFrame | None:
    path = PATHS[key]
    if not path.is_file():
        print(f"skip {key}: missing {path.name}")
        return None
    return pd.read_csv(path)


rank_overall = _load_csv("ranking_overall")
rank_slice = _load_csv("ranking_by_slice")
rank_person = _load_csv("ranking_personalization")

run_meta = None
if PATHS["run_meta"].is_file():
    run_meta = json.loads(PATHS["run_meta"].read_text(encoding="utf-8"))
    print("pool_methods:", run_meta.get("pool_methods"))
    print("ranker_methods:", run_meta.get("ranker_methods"))
    print("catalog_methods:", run_meta.get("catalog_methods"))
    print("pools_jsonl:", run_meta.get("pools_jsonl"))
    print(
        f"k_final={run_meta.get('k_final')} k_retrieval={run_meta.get('k_retrieval')} "
        f"k_personalization={run_meta.get('k_personalization')}"
    )
else:
    print("no eval_ranking_run_meta.json")

pool_methods: ['two_tower_v1']
ranker_methods: ['two_tower_v1_heuristic_logpop_blend']
catalog_methods: ['popularity_train']
pools_jsonl: /home/ryanr/workspace/steam_recommendations/artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl
k_final=10 k_retrieval=100 k_personalization=10


In [7]:
display(Markdown("### Ranking overall (sorted by NDCG@K)"))
display(
    Markdown(
        "Methods: **catalog** (`popularity_train` = global pop scores) · "
        "**pool** (`two_tower_v1` = frozen pool order) · "
        "**ranker** (`*_heuristic_*` = rerank within pool)"
    )
)
if rank_overall is not None:
    display(rank_overall.sort_values("NDCG@K", ascending=False))

display(Markdown("### Personalization (standalone, same as appended cols on overall)"))
if rank_person is not None:
    display(rank_person.sort_values("PersonalizationGapVsPopularity@10", ascending=False))

### Ranking overall (sorted by NDCG@K)

Methods: **catalog** (`popularity_train` = global pop scores) · **pool** (`two_tower_v1` = frozen pool order) · **ranker** (`*_heuristic_*` = rerank within pool)

,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,OracleHit@K,OracleNDCG@K,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,two_tower_v1_heuristic_logpop_blend,0.19328,0.019560,0.184095,0.064321,0.092892,0.067059,0.51224,0.498310,0.204632,0.501587,6.272161,0.720097
1,popularity_train,0.15112,0.015184,0.146796,0.050594,0.073109,0.052221,0.76280,0.752102,0.212335,0.034921,5.317471,0.000000
2,two_tower_v1,0.04680,0.004728,0.043740,0.010325,0.018161,0.011008,0.51224,0.498310,0.261639,0.987302,12.167556,0.995623


### Personalization (standalone, same as appended cols on overall)

,method,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
1,two_tower_v1,0.261639,0.987302,12.167556,0.995623
2,two_tower_v1_heuristic_logpop_blend,0.204632,0.501587,6.272161,0.720097
0,popularity_train,0.212335,0.034921,5.317471,0.000000


In [4]:
display(Markdown("### By slice"))
if rank_slice is not None:
    display(rank_slice.sort_values(["slice_name", "NDCG@K"], ascending=[True, False]))

display(Markdown("### By support bucket"))
rank_support = _load_csv("ranking_by_support")
if rank_support is not None:
    display(rank_support.sort_values(["train_support_bucket", "NDCG@K"], ascending=[True, False]))

### By slice

,slice_name,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,OracleHit@K,OracleNDCG@K,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,slice_a_multi_target,two_tower_v1_heuristic_logpop_blend,0.271724,0.031172,0.113367,0.034181,0.068322,0.081396,0.773793,0.533618,0.203806,0.412698,6.504750,0.793487
1,slice_a_multi_target,popularity_train,0.128276,0.014069,0.053728,0.019418,0.035444,0.047469,0.892414,0.707962,0.212240,0.034921,5.312688,0.000000
2,slice_a_multi_target,two_tower_v1,0.091034,0.009931,0.038280,0.009408,0.020537,0.021192,0.773793,0.533618,0.256915,0.933333,12.163190,0.997354
3,slice_b_single_target,two_tower_v1_heuristic_logpop_blend,0.188450,0.018845,0.188450,0.066176,0.094404,0.066176,0.496136,0.496136,0.204683,0.501587,6.257840,0.715579
4,slice_b_single_target,popularity_train,0.152527,0.015253,0.152527,0.052514,0.075428,0.052514,0.754820,0.754820,0.212341,0.034921,5.317765,0.000000
5,slice_b_single_target,two_tower_v1,0.044076,0.004408,0.044076,0.010381,0.018015,0.010381,0.496136,0.496136,0.261930,0.987302,12.167824,0.995516


### By support bucket

,train_support_bucket,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,OracleHit@K,OracleNDCG@K,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,0,two_tower_v1_heuristic_logpop_blend,0.211200,0.021120,0.211040,0.078975,0.109642,0.079055,0.504000,0.504000,0.204387,0.463492,6.162312,0.685295
1,0,popularity_train,0.174720,0.017472,0.174720,0.061863,0.087912,0.061863,0.760960,0.760960,0.212432,0.034921,5.319178,0.000000
2,0,two_tower_v1,0.049920,0.004992,0.049760,0.011584,0.020272,0.011638,0.504000,0.504000,0.262590,0.965079,12.187545,0.994675
3,1,two_tower_v1_heuristic_logpop_blend,0.208960,0.020896,0.208960,0.074154,0.105206,0.074154,0.511680,0.511680,0.205091,0.466667,6.216828,0.703931
4,1,popularity_train,0.181120,0.018112,0.181120,0.062714,0.089706,0.062714,0.786560,0.786560,0.212320,0.034921,5.319652,0.000000
5,1,two_tower_v1,0.044800,0.004480,0.044800,0.010378,0.018183,0.010378,0.511680,0.511680,0.263031,0.980952,12.161975,0.995439
6,2-3,two_tower_v1_heuristic_logpop_blend,0.176128,0.017613,0.176128,0.060125,0.086892,0.060125,0.487953,0.487699,0.204689,0.473016,6.294136,0.726390
7,2-3,popularity_train,0.136804,0.013680,0.136666,0.045143,0.066066,0.045163,0.750762,0.750524,0.212267,0.034921,5.316660,0.000000
8,2-3,two_tower_v1,0.039878,0.003988,0.039740,0.009446,0.016265,0.009492,0.487953,0.487699,0.261712,0.974603,12.159349,0.995639
9,4-7,two_tower_v1_heuristic_logpop_blend,0.176961,0.018795,0.133646,0.041064,0.066684,0.053940,0.555892,0.490258,0.204301,0.457143,6.437694,0.771843


In [5]:
if rank_overall is not None:
    cols = [
        "method",
        "Hit@K",
        "NDCG@K",
        "MRR",
        "OracleNDCG@K",
        "PersonalizationGapVsPopularity@10",
    ]
    cols = [c for c in cols if c in rank_overall.columns]
    display(Markdown("### Compact comparison (relevance + gap vs popularity)"))
    display(rank_overall[cols].sort_values("NDCG@K", ascending=False))

### Compact comparison (relevance + gap vs popularity)

,method,Hit@K,NDCG@K,MRR,OracleNDCG@K,PersonalizationGapVsPopularity@10
0,two_tower_v1_heuristic_logpop_blend,0.19328,0.092892,0.067059,0.498310,0.720097
1,popularity_train,0.15112,0.073109,0.052221,0.752102,0.000000
2,two_tower_v1,0.04680,0.018161,0.011008,0.498310,0.995623
